# Phase 2: Stylometric Feature Engineering & Statistical Analysis (RQ 1.2)

This notebook extracts **44 interpretable stylometric, linguistic, and structural features** across our balanced English corpus ($N = 5,236$ tracks: $2,618$ TikTok-Viral vs. $2,618$ Non-Viral commercial tracks released between 2020 and 2026).

### Literature Grounding
- **Lexical, Linguistic & Syntactic Baseline:** Replicates the validated feature inventory of **Zangerle et al. [2020]** and **Martín-Gutiérrez et al. [2023]** (token counts, TTR, hapax/dis/tris-legomena, stop words, lemma ratio, and past-tense density), while consciously excluding affective VADER sentiment to focus strictly on structural virality.
- **Repetition & Short-Form Compression:** Replicates the `zlib` information compressibility and line-repetition framework of **Parada-Cabaleiro et al. [2024]**.
- **Phonetic Rhyme Pacing:** CMU Pronouncing Dictionary rhyme modeling to capture platform-native metric structures.

### Statistical Validation Pipeline
1. **95% Confidence Intervals** ($\bar{x} \pm 1.96 \cdot \text{SE}$) & Cohen's $d$ effect sizes with **Bonferroni and Benjamini-Hochberg FDR corrections**.
2. **Spearman Correlation Heatmap** to assess multicollinearity across the 5 feature groups.
3. **Chi-Square ($\chi^2$) Test of Independence** ($2 \times 2$ contingency tables) fitted strictly on the **Training Set** to prevent data leakage.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.build_features import (
    ALL_FEATURE_COLUMNS,
    FEATURE_GROUPS,
    run_feature_pipeline,
)
from src.features.statistical_tests import (
    compute_chi_square_vocabulary,
    compute_feature_confidence_intervals,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
print(f"Loaded Phase 2 pipeline ({len(ALL_FEATURE_COLUMNS)} features across {len(FEATURE_GROUPS)} groups).")

## 1. Extract All 44 Interpretable Features Across 5 Groups
Computes lexical diversity, repetition/compression, pronoun/function words, `spaCy` POS/lemma/tense, and CMU phonetic rhyme extractors on verified English tracks.

In [ ]:
splits = run_feature_pipeline(force_recompute=False)
features_full = splits["full"]
features_train = splits["train"]
features_test = splits["test"]

features_full[["track_name", "primary_artist", "label_name"] + ALL_FEATURE_COLUMNS[:10]].head()

## 2. 95% Confidence Intervals, Cohen's $d$ & Multiple Testing Corrections (RQ 1.2)
Computes group means, 95% Confidence Intervals ($\bar{x} \pm 1.96 \cdot \text{SE}$), Cohen's $d$, Welch's $t$-test $p$-values, Bonferroni-adjusted $p$-values, and Benjamini-Hochberg False Discovery Rate (FDR) $q$-values.

In [ ]:
ci_df = compute_feature_confidence_intervals(
    df=features_full,
    feature_groups=FEATURE_GROUPS,
    label_col="label",
)

ci_out_path = PROJECT_ROOT / "data" / "processed" / "stat_ci_summary.csv"
ci_df.to_csv(ci_out_path, index=False)
print(f"Saved 95% CI summary table with Bonferroni & FDR corrections to {ci_out_path}")

display_cols = [
    "feature_group",
    "feature",
    "viral_mean",
    "non_viral_mean",
    "diff_mean",
    "cohens_d",
    "p_value",
    "bonferroni_p",
    "fdr_bh_p",
    "significant_bonferroni",
]
ci_df[display_cols].head(25)

## 3. Visualizing Standardized Effect Sizes (Top Features by Cohen's $d$)

In [ ]:
top_feats = ci_df.head(18).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#d95f02" if d > 0 else "#1b9e77" for d in top_feats["cohens_d"]]
ax.barh(top_feats["feature"], top_feats["cohens_d"], color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Cohen's d (Positive = Higher in TikTok Viral, Negative = Higher in Non-Viral)", fontsize=11)
ax.set_title("Top Stylometric Features Distinguishing TikTok-Viral from Non-Viral Lyrics", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Multicollinearity Inspection (Feature Correlation Heatmap)

In [ ]:
top_feature_names = ci_df.head(15)["feature"].tolist()
corr_matrix = features_full[top_feature_names].corr(method="spearman")

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.title("Spearman Correlation Across Top Stylometric Features", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Chi-Square ($\chi^2$) Test of Independence (Fitted on Training Split Only)
To adhere strictly to Text Mining best practices and prevent test-set leakage, vocabulary testing is performed strictly on `features_train` ($N = 4,188$ tracks).

In [ ]:
chi2_df = compute_chi_square_vocabulary(
    df=features_train,
    text_col="lyrics",
    label_col="label",
    min_doc_freq=25,
    top_k=100,
)

chi2_out_path = PROJECT_ROOT / "data" / "processed" / "stat_chi2_vocab_train.csv"
chi2_df.to_csv(chi2_out_path, index=False)

print("--- Top 15 Words Associated with TikTok-Viral Songs in Training Set ---")
display(chi2_df[chi2_df["direction"] == "Viral (label=1)"].head(15))

print("\n--- Top 15 Words Associated with Non-Viral Songs in Training Set ---")
display(chi2_df[chi2_df["direction"] == "Non-Viral (label=0)"].head(15))